In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

agent= create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [19]:
config={"configurable":{"thread_id":"test-1"}}

In [20]:
questions=[
"what is 2+2?",
"what is 4+73",
"what is 283*23",
"what is 10-3",
"what is 2739*8",
"what is 39-28?",
"what is 2823-29"
]

In [21]:
for q in questions:
    res=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages:{res}")
    print(f"messages:{len(res['messages'])}")

messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='6650d286-602e-41bf-8eb8-3db28208f175'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.008931564, 'completion_tokens_details': None, 'prompt_time': 0.003003384, 'prompt_tokens_details': None, 'queue_time': 0.006032541, 'total_time': 0.011934948}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3b93-7888-79a2-998c-edb245a46a5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
messages:2
messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='6650d286-602e-41bf-8eb8-3db28208f175'), AIMessage(co

In [24]:
from langchain_core.tools import tool

@tool
def search_hotel(city:str) -> str:
    """ Search hotels - returns long response to use more tokens"""
    return f""" Hotels in {city}:
    1.Grand Hotel - 5star, $350/night, spa, pool, gym
    2.City Inn - 4 star,$180/night, business center
    3.Budget stay - 3star,$75/night,free wifi

"""

agent= create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

In [25]:
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [26]:
cities=["paris", "tokyo","london","newyork","dubai","Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

paris: ~832 tokens, 2 messages
[HumanMessage(content='Find hotels in paris', additional_kwargs={}, response_metadata={}, id='00ab99bf-fe6c-44e7-835c-09f6dd6b71fa'), AIMessage(content="Paris, the City of Light, is a popular destination for travelers from around the world. Here are some top-rated hotels in Paris:\n\n**Luxury Hotels**\n\n1. **The Ritz Paris** - A 5-star hotel located on the prestigious Place Vendôme, offering elegant rooms and exceptional service.\n2. **Hotel Plaza Athenee** - A 5-star hotel located on the prestigious Avenue Montaigne, offering luxurious rooms and a world-class spa.\n3. **Le Bristol Paris** - A 5-star hotel located in the heart of Paris, offering elegant rooms and a Michelin-starred restaurant.\n4. **Four Seasons Hotel George V Paris** - A 5-star hotel located on the Avenue George V, offering luxurious rooms and a world-class spa.\n5. **Shangri-La Hotel Paris** - A 5-star hotel located in the former home of Prince Roland Bonaparte, offering luxurious room

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kk2j6z6xe9qbh8zcaj2zazca` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5298, Requested 872. Please try again in 1.7s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [37]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
@tool
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"
@tool
def send_email_tool(recipient:str, sub:str,body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{sub}'."

In [39]:
agent= create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    tools=[read_email_tool,send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]   
                },
                "read_email_tool":False
            }
        )
    ]
)

In [40]:
config={"configurable":{"thread_id":"test-approve"}}

result=agent.invoke(
    {
        "messages":[HumanMessage(content="send an email to john@test.com with subject 'Hello john' and body 'How are you?")]
    },
    config=config
)

In [41]:
result

{'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello john' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='655ebcb2-1412-44c2-aadc-5e2612303930'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h46nh1zb7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","sub":"Hello john"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 311, 'total_tokens': 342, 'completion_time': 0.03502108, 'completion_tokens_details': None, 'prompt_time': 0.021016599, 'prompt_tokens_details': None, 'queue_time': 0.168602014, 'total_time': 0.056037679}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3bca-e2ea-7951-8083-0222a2dc40e0-0', tool_calls=[{'name': 'send_email_tool', 'args':

In [43]:
from langgraph.types import  Command
if '__interrupt__' in result:
    print("paused! Approving...")
    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

paused! Approving...
Result:This is the result of sending the email.


In [44]:
result

{'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello john' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='655ebcb2-1412-44c2-aadc-5e2612303930'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h46nh1zb7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","sub":"Hello john"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 311, 'total_tokens': 342, 'completion_time': 0.03502108, 'completion_tokens_details': None, 'prompt_time': 0.021016599, 'prompt_tokens_details': None, 'queue_time': 0.168602014, 'total_time': 0.056037679}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3bca-e2ea-7951-8083-0222a2dc40e0-0', tool_calls=[{'name': 'send_email_tool', 'args':